In [ ]:
import pandas as pd

xwalk = pd.read_excel('../data/datasets/soc_2010_to_2018_crosswalk.xlsx', header=8)
xwalk['2010 SOC Code'] = xwalk['2010 SOC Code'].astype(str).str.strip()
xwalk['2018 SOC Code'] = xwalk['2018 SOC Code'].astype(str).str.strip()

# Flag footnote markers, then strip them from title text
xwalk['has_footnote_marker'] = (
    xwalk['2010 SOC Title'].str.contains(r'\(#\)', na=False) |
    xwalk['2018 SOC Title'].str.contains(r'\(##\)', na=False)
)
xwalk['2010 SOC Title'] = xwalk['2010 SOC Title'].str.replace(r'\s*\(#\)', '', regex=True)
xwalk['2018 SOC Title'] = xwalk['2018 SOC Title'].str.replace(r'\s*\(##\)', '', regex=True)

# Classify match type (1:1, 1:many split, many:1 merge, many:many tangled)
counts_2010 = xwalk.groupby('2010 SOC Code').size()
counts_2018 = xwalk.groupby('2018 SOC Code').size()

def classify(row):
    n_2010 = counts_2010[row['2010 SOC Code']]
    n_2018 = counts_2018[row['2018 SOC Code']]
    if n_2010 == 1 and n_2018 == 1:
        return '1:1'
    elif n_2010 > 1 and n_2018 == 1:
        return '1:many'
    elif n_2010 == 1 and n_2018 > 1:
        return 'many:1'
    else:
        return 'many:many'

xwalk['match_type'] = xwalk.apply(classify, axis=1)

# Manual overrides for tangled clusters resolved in crosswalk_readme.md —
# marks the TRUE handling rule per row, since match_type alone
# mis-labels resolved hub/split clusters as one blob
manual_handling = {
    ('11-9199', '13-1082'): 'merge_slice',
    ('13-1199', '13-1082'): 'merge_slice',
    ('15-1199', '13-1082'): 'merge_slice',
    ('39-1021', '53-1044'): 'merge_slice',
    ('53-1031', '53-1044'): 'merge_slice',
    ('29-2071', '29-9021'): 'merge_slice',
    ('29-9099', '29-9021'): 'merge_slice',
    ('53-3022', '53-3053'): 'merge_slice',
    ('53-3041', '53-3053'): 'merge_slice',
    # add remaining rows from crosswalk_readme.md as they're confirmed
}

def assign_handling(row):
    key = (row['2010 SOC Code'], row['2018 SOC Code'])
    if key in manual_handling:
        return manual_handling[key]
    return row['match_type']

xwalk['handling'] = xwalk.apply(assign_handling, axis=1)

xwalk.to_csv('../data/datasets/processed/crosswalk_clean.csv', index=False)
print(f"Saved {len(xwalk)} rows. Handling breakdown:")
print(xwalk['handling'].value_counts())